## 1、TextSplitter的使用
### 1、使用细节
### （1）使用TextSplitter作为各种具体文档拆分器的父类
### （2）内部常用属性：
#### 参数：
#### chunk_size: 返回块的最大尺寸，单位是字符数。默认值为4000（由长度函数测量）
#### chunk_overlap: 相邻两个块之间的字符重叠数,避免信息在边界处被切断而丢失。默认值为200,通常会设置为chunk_size的10% - 20%。
#### length_function: 用于测量给定块字符数的函数。默认赋值为len函数。len函数在Python中按 Unicode字符计数，所以一个汉字、一个英文字母、一个符号都算一个字符。
#### keep_separator: 是否在块中保留分隔符，默认值为False
#### add_start_index: 如果为 `True`，则在元数据中包含块的起始索引。默认值为False
#### strip_whitespace: 如果为 `True`，则从每个文档的开始和结束处去除空白字符。默认值为True
"""

② 内部定义的常用的方法：

### 方式1：传入的参数类型：字符串; 返回值类型：List[str]
split_text(xxx)
### 方式2：传入的参数类型：List[str]; 返回值类型：List[Document]
create_documents(xxx) #底层调用了split_text(xxx)
### 方式3：传入的参数类型：List[Document]; 返回值类型：List[Document]
split_documents(xxx) #底层调用了create_documents()

##  具体的拆分器1：CharacterTextSplitter：Split by character

### 参数说明：
#### chunk_size ：每个切块的最大token数量，默认值为4000。
#### chunk_overlap ：相邻两个切块之间的最大重叠token数量，默认值为200。
#### separator ：分割使用的分隔符，默认值为"\n\n"。
#### length_function ：用于计算切块长度的方法。默认赋值为父类TextSplitter的len函数


举例1：体会：chunk_size、chunk_overlap，字符串文本的分割

In [3]:
# 1.导入相关依赖
from langchain_text_splitters import CharacterTextSplitter

# 2.示例文本
text = """
LangChain 是一个用于开发由语言模型驱动的应用程序的框架的。它提供了一套工具和抽象，使开发者能够更容易地构建复杂的应用程序。
"""

# 3.定义字符分割器
splitter = CharacterTextSplitter(
    chunk_size=51, # 每块大小
    chunk_overlap=7,# 块与块之间的重复字符数
    #length_function=len,
    separator=""   # 设置为空字符串时，表示禁用分隔符
)

# 4.分割文本
texts = splitter.split_text(text)
print(type(texts))
print(texts)
# 5.打印结果
for i, chunk in enumerate(texts):
    print(f"块 {i+1}:长度：{len(chunk)}")
    print(chunk)
    print("-" * 50)

<class 'list'>
['LangChain 是一个用于开发由语言模型驱动的应用程序的框架的。它提供了一套工具和抽象，使开发者', '抽象，使开发者能够更容易地构建复杂的应用程序。']
块 1:长度：50
LangChain 是一个用于开发由语言模型驱动的应用程序的框架的。它提供了一套工具和抽象，使开发者
--------------------------------------------------
块 2:长度：23
抽象，使开发者能够更容易地构建复杂的应用程序。
--------------------------------------------------


举例2：体会separator指定分割符
#####  separator优先原则：当设置了 separator （如"。"），分割器会首先尝试在分隔符处分割，然后再考虑 chunk_size。这是为了避免在句子中间硬性切断

In [1]:
# 1.导入相关依赖
from langchain_text_splitters import CharacterTextSplitter

# 2.定义要分割的文本
text = "这是一个示例文本啊。我们将使用CharacterTextSplitter将其分割成小块。分割基于字符数。"

# text = """
# LangChain 是一个用于开发由语言模型。驱动的应用程序的框架的。它提供了一套工具和抽象。使开发者能够更容易地构建复杂的应用程序。
# """

# 3.定义分割器实例
text_splitter = CharacterTextSplitter(
    # chunk_size=30,   # 每个块的最大字符数
    chunk_size=43,   # 每个块的最大字符数
    chunk_overlap=0, # 块之间的重叠字符数
    separator="。",  # 按句号分割 （分隔符优先）
)

# 4.开始分割
chunks = text_splitter.split_text(text)

# 5.打印效果
for  i,chunk in enumerate(chunks):
    print(f"块 {i + 1}:长度：{len(chunk)}")
    print(chunk)
    print("-"*50)


块 1:长度：43
这是一个示例文本啊。我们将使用CharacterTextSplitter将其分割成小块
--------------------------------------------------
块 2:长度：7
分割基于字符数
--------------------------------------------------


In [5]:
# 1.导入相关依赖
from langchain_text_splitters import CharacterTextSplitter

# 2.定义要分割的文本
text = "这是一个示例文本啊。我们将使用CharacterTextSplitter将其分割成小块。分割基于字符数。"

# text = """
# LangChain 是一个用于开发由语言模型。驱动的应用程序的框架的。它提供了一套工具和抽象。使开发者能够更容易地构建复杂的应用程序。
# """

# 3.定义分割器实例
text_splitter = CharacterTextSplitter(
    chunk_size=43,   # 每个块的最大字符数,合并两个句号的句子
    chunk_overlap=0, # 块之间的重叠字符数
    separator="。",  # 按句号分割 （分隔符优先）
)

# 4.开始分割
chunks = text_splitter.split_text(text)

# 5.打印效果
for  i,chunk in enumerate(chunks):
    print(f"块 {i + 1}:长度：{len(chunk)}")
    print(chunk)
    print("-"*50)


块 1:长度：43
这是一个示例文本啊。我们将使用CharacterTextSplitter将其分割成小块
--------------------------------------------------
块 2:长度：7
分割基于字符数
--------------------------------------------------


### 举例3：熟悉 keep_separator、separator 以及 chunk_overlap何时生效
### separator先用 。 把文本切成小片段，然后再把这些小片段重新合并成不超过 chunk_size 的大块。

In [6]:
# 1.导入相关依赖
from langchain_text_splitters import CharacterTextSplitter

# 2.定义要分割的文本
text = "这是第一段文本。这是第二段内容。最后一段结束。"

# 3.定义字符分割器
text_splitter = CharacterTextSplitter(
    separator="。",
    chunk_size=20,
    chunk_overlap=8,
    keep_separator=True #chunk中是否保留切割符:'。'
)

# 4.分割文本
chunks = text_splitter.split_text(text)

# 5.打印结果
for  i,chunk in enumerate(chunks):
    print(f"块 {i + 1}:长度：{len(chunk)}")
    print(chunk)
    print("-"*50)

块 1:长度：15
这是第一段文本。这是第二段内容
--------------------------------------------------
块 2:长度：16
。这是第二段内容。最后一段结束。
--------------------------------------------------


## 3、具体的拆分器2：RecursiveCharacterTextSplitter：最常用
#### separators or ["\n\n", "\n", " ", ""]递归拆分  seperaters如果是空就取  ["\n\n", "\n", " ", ""]
举例1：使用split_text()方法演示

LangChain框
-------
架特性
-------
多模型集成(GPT
-------
/Claude)
-------
记忆管理功能
-------
链式调用设计。文档
-------
分析场景示例：需要处
-------
理PDF/Word等
-------
格式。
-------


举例2：使用create_documents()方法演示，传入字符串列表，返回Document对象列表，开启索引

In [3]:
# 2.定义RecursiveCharacterTextSplitter分割器对象
## RecursiveCharacterTextSplitter 的核心规则是：按一组“分隔符优先级”从大到小递归切分，尽量让每个 chunk 长度不超过 chunk_size。
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=10,
    chunk_overlap=0,
    add_start_index=True,##当前chunk首字符在源文件的索引
)

# 3.定义拆分的内容
text_list = ["LangChain框架特性\n\n多模型集成(GPT/Claude)\n记忆管理功能\n链式调用设计。文档分析场景示例：需要处理PDF/Word等格式。"]

# 4.拆分器分割
paragraphs = text_splitter.create_documents(text_list)
print(type(paragraphs))
print(paragraphs)
print("-"*50)
for para in paragraphs:
    print(para)
    print('-------')

<class 'list'>
[Document(metadata={'start_index': 0}, page_content='LangChain框'), Document(metadata={'start_index': 10}, page_content='架特性'), Document(metadata={'start_index': 15}, page_content='多模型集成(GPT'), Document(metadata={'start_index': 24}, page_content='/Claude)'), Document(metadata={'start_index': 33}, page_content='记忆管理功能'), Document(metadata={'start_index': 40}, page_content='链式调用设计。文档'), Document(metadata={'start_index': 49}, page_content='分析场景示例：需要处'), Document(metadata={'start_index': 59}, page_content='理PDF/Word等'), Document(metadata={'start_index': 69}, page_content='格式。')]
--------------------------------------------------
page_content='LangChain框' metadata={'start_index': 0}
-------
page_content='架特性' metadata={'start_index': 10}
-------
page_content='多模型集成(GPT' metadata={'start_index': 15}
-------
page_content='/Claude)' metadata={'start_index': 24}
-------
page_content='记忆管理功能' metadata={'start_index': 33}
-------
page_content='链式调用设计。文档' metadata={'start_index': 40}

举例3：使用create_documents()方法演示，将本地文件内容加载成字符串，进行拆分

In [9]:
# 2.打开.txt文件
with open("asset/load/08-ai.txt", encoding="utf-8") as f:
    state_of_the_union = f.read()  #返回的是字符串

print(type(state_of_the_union))  # <class 'str'>

# 3.定义RecursiveCharacterTextSplitter（递归字符分割器）
## RecursiveCharacterTextSplitter 的核心规则是：按一组“分隔符优先级”从大到小递归切分，尽量让每个 chunk 长度不超过 chunk_size。
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=100,
    chunk_overlap=20,
    #chunk_overlap=0,
    length_function=len
)

# 4.分割文本
texts = text_splitter.create_documents([state_of_the_union])  #字符串用【】包起来

# 5.打印分割文本
for document in texts:
    print(f"🔥{document.page_content}")

<class 'str'>
🔥人工智能（AI）是什么？
🔥人工智能（Artificial
🔥Intelligence，简称AI）是指由计算机系统模拟人类智能的技术，使其能够执行通常需要人类认知能力的任务，如学习、推理、决策和语言理解。AI的核心目标是让机器具备感知环境、处理信息并自主行动的
🔥让机器具备感知环境、处理信息并自主行动的能力。
🔥1. AI的技术基础
AI依赖多种关键技术：

机器学习（ML）：通过算法让计算机从数据中学习规律，无需显式编程。例如，推荐系统通过用户历史行为预测偏好。
🔥深度学习：基于神经网络的机器学习分支，擅长处理图像、语音等复杂数据。AlphaGo击败围棋冠军便是典型案例。

自然语言处理（NLP）：使计算机理解、生成人类语言，如ChatGPT的对话能力。
🔥2. AI的应用场景
AI已渗透到日常生活和各行各业：

医疗：辅助诊断（如AI分析医学影像）、药物研发加速。

交通：自动驾驶汽车通过传感器和AI算法实现安全导航。
🔥金融：欺诈检测、智能投顾（如风险评估模型）。

教育：个性化学习平台根据学生表现调整教学内容。

3. AI的挑战与未来
尽管前景广阔，AI仍面临问题：
🔥伦理争议：数据隐私、算法偏见（如招聘AI歧视特定群体）。

就业影响：自动化可能取代部分人工岗位，但也会创造新职业。

技术瓶颈：通用人工智能（AGI）尚未实现，当前AI仅擅长特定任务。
🔥未来，AI将与人类协作而非替代：医生借助AI提高诊断效率，教师利用AI定制课程。其发展需平衡技术创新与社会责任，确保技术造福全人类。


举例4：使用split_documents()方法演示，利用PDFLoader加载文档，对文档的内容用递归切割器切割

In [7]:
# 1.导入相关依赖
from langchain_community.document_loaders import PyPDFLoader

# 2.定义PyPDFLoader加载器
loader = PyPDFLoader("./asset/load/02-load.pdf")

# 3.加载和切割文档对象
docs = loader.load()   # 返回Document对象构成的list
# print(f"第0页：\n{docs[0]}")

# 4.定义切割器
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    # chunk_size=120,
    chunk_overlap=0,
    # chunk_overlap=100,
    length_function=len,
    ##在每个切分后的 Document 的 metadata 里，记录这个 chunk 在原文中的起始字符位置。它主要用于：split_documents(),create_documents()
    add_start_index=True,

)

# 5.对pdf内容进行切割得到文档对象
paragraphs = text_splitter.split_documents(docs)
#paragraphs = text_splitter.create_documents([text])
print(paragraphs)
for para in paragraphs:
    print(para.page_content)
    print('-------')

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2025-06-20T17:18:19+08:00', 'moddate': '2025-06-20T17:18:19+08:00', 'source': './asset/load/02-load.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'start_index': 0}, page_content='"他的车，他的命！ 他忽然想起来，一年，二年，至少有三四年；一滴汗，两滴汗，不\n知道多少万滴汗，才挣出那辆车。从风里雨里的咬牙，从饭里茶里的自苦，才赚出那辆车。\n那辆车是他的一切挣扎与困苦的总结果与报酬，像身经百战的武士的一颗徽章。……他老想\n着远远的一辆车，可以使他自由，独立，像自己的手脚的那么一辆车。" \n \n"他吃，他喝，他嫖，他赌，他懒，他狡猾， 因为他没了心，他的心被人家摘了去。他'), Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2025-06-20T17:18:19+08:00', 'moddate': '2025-06-20T17:18:19+08:00', 'source': './asset/load/02-load.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'start_index': 198}, page_content='只剩下那个高大的肉架子，等着溃烂，预备着到乱死岗子去。……体面的、要强的、好梦想\n的、利己的、个人的、健壮的、伟大的祥子，不知陪着人家送了多少回殡；不知道何时何地\n会埋起他自己来， 埋起这堕落的、 自私的、 不幸的、 社会病胎里的产儿， 个人主义的末路鬼！\n"')]
"他的车，他的命！ 他忽然想起来，一年，二年，至少有三四年；一滴汗，两滴汗，不
知道多少万滴汗，才挣出那辆车。从风里雨里的咬牙，从饭

## 具体的拆分器3：TokenTextSplitter / CharacterTextSplitter

#### 举例1：使用TokenTextSplitter

In [8]:
import tiktoken
from langchain_text_splitters import TokenTextSplitter

encoding = tiktoken.get_encoding("cl100k_base")

text_splitter = TokenTextSplitter(
    chunk_size=33,
    chunk_overlap=0,
    encoding_name="cl100k_base",
)

text = "人工智能是一个强大的开发框架。它支持多种语言模型和工具链。人工智能是指通过计算机程序模拟人类智能的一门科学。自20世纪50年代诞生以来，人工智能经历了多次起伏。"

texts = text_splitter.split_text(text)

print(f"原始文本被分割成了 {len(texts)} 个块:")

for i, chunk in enumerate(texts):
    token_count = len(encoding.encode(chunk))
    print(f"块 {i+1}: 字符长度：{len(chunk)} token长度：{token_count}")
    print(chunk)
    print("-" * 50)

原始文本被分割成了 3 个块:
块 1: 字符长度：29 token长度：33
人工智能是一个强大的开发框架。它支持多种语言模型和工具链。
--------------------------------------------------
块 2: 字符长度：32 token长度：33
人工智能是指通过计算机程序模拟人类智能的一门科学。自20世纪50
--------------------------------------------------
块 3: 字符长度：19 token长度：22
年代诞生以来，人工智能经历了多次起伏。
--------------------------------------------------


#### 举例2：使用CharacterTextSplitter

In [10]:
# 1.导入相关依赖
from langchain_text_splitters import CharacterTextSplitter
import tiktoken  # 用于计算Token数量


# 2.定义通过Token切割器
# CharacterTextSplitter.from_tiktoken_encoder（）优先按你指定的 separator="。" 切分文本，然后再尝试按 chunk_size 合并；
# 但是如果某一个按句号切出来的片段本身已经超过 chunk_size=10，它不会继续强制把这个片段再切小。
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base", # 使用 OpenAI 的编码器
    chunk_size=10, #设置最大的token数
    chunk_overlap=0,
    separator="。",  # 指定中文句号为分隔符
    keep_separator=False,  # chunk中是否保留分隔符
)
# 3.定义文本
text = "人工智能是一个强大的开发框架。它支持多种语言模型和工具链。今天天气很好，想出去踏青。但是又比较懒不想出去，怎么办"

# 4.开始切割
texts = text_splitter.split_text(text)
print(f"分割后的块数: {len(texts)}")

# 5.初始化tiktoken编码器（用于Token计数）
encoder = tiktoken.get_encoding("cl100k_base")  # 确保与CharacterTextSplitter的encoding_name一致

# 6.打印每个块的Token数和内容
for i, chunk in enumerate(texts):
    tokens = encoder.encode(chunk)  # 现在encoder已定义
    print(f"块 {i + 1}: {len(tokens)} Token\n内容: {chunk}\n")


Created a chunk of size 17, which is longer than the specified 10
Created a chunk of size 14, which is longer than the specified 10
Created a chunk of size 18, which is longer than the specified 10


分割后的块数: 4
块 1: 17 Token
内容: 人工智能是一个强大的开发框架

块 2: 14 Token
内容: 它支持多种语言模型和工具链

块 3: 18 Token
内容: 今天天气很好，想出去踏青

块 4: 21 Token
内容: 但是又比较懒不想出去，怎么办



In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken

##RecursiveCharacterTextSplitter.from_tiktoken_encoder()
##既用 token 数控制 chunk_size，又会递归使用 separators 继续切分，直到每个 chunk 不超过 10 token。
"""
1. 先按中文句号 "。" 切
2. 如果某一段仍然超过 10 token，再按中文逗号 "，" 切
3. 如果还超过 10 token，再按空格 " " 切
4. 如果还超过 10 token，最后按字符 "" 硬切
"""
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",
    chunk_size=10,
    chunk_overlap=0,
    separators=["。", "，", " ", ""],
    keep_separator=False,
)

text = "人工智能是一个强大的开发框架。它支持多种语言模型和工具链。今天天气很好，想出去踏青。但是又比较懒不想出去，怎么办"

texts = text_splitter.split_text(text)

encoder = tiktoken.get_encoding("cl100k_base")

for i, chunk in enumerate(texts):
    tokens = encoder.encode(chunk)
    print(f"块 {i + 1}: {len(tokens)} Token")
    print(chunk)
    print("-" * 50)

块 1: 9 Token
人工智能是一个强
--------------------------------------------------
块 2: 8 Token
大的开发框架
--------------------------------------------------
块 3: 10 Token
它支持多种语言模型
--------------------------------------------------
块 4: 4 Token
和工具链
--------------------------------------------------
块 5: 8 Token
今天天气很好
--------------------------------------------------
块 6: 9 Token
想出去踏青
--------------------------------------------------
块 7: 10 Token
但是又比较懒
--------------------------------------------------
块 8: 5 Token
不想出去
--------------------------------------------------
块 9: 5 Token
怎么办
--------------------------------------------------


## 体的拆分器4：SemanticChunker：语义分块
["百分位数", "标准差", "四分位距", "梯度"]
#### 文本的语义结构 进行智能分块，使每个分块保持 语义完整性 ，从而提高检索增强生成(RAG)等应用的效果。

In [14]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai.embeddings import OpenAIEmbeddings
import os
import dotenv

dotenv.load_dotenv()

# 加载文本
with open("asset/load/09-ai1.txt", encoding="utf-8") as f:
    state_of_the_union = f.read()  #返回字符串

# 获取嵌入模型
os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")
embed_model = OpenAIEmbeddings(
    model="text-embedding-3-large"
)

# 获取切割器
text_splitter = SemanticChunker(
    embeddings=embed_model,
    breakpoint_threshold_type="percentile",
    #断点阈值类型：字面值["百分位数", "标准差", "四分位距", "梯度"] 选其一[percentile,standard_deviation,interquartile,gradient]见PDF P21
    breakpoint_threshold_amount=65.0 #断点阈值数量 (极低阈值 → 高分割敏感度)
)

# 切分文档
docs = text_splitter.create_documents(texts = [state_of_the_union])
print(len(docs))
for doc in docs:
    print(f"🔍 文档 {doc}:")

4
🔍 文档 page_content='人工智能综述：发展、应用与未来展望

摘要
人工智能（Artificial Intelligence，AI）作为计算机科学的一个重要分支，近年来取得了突飞猛进的发展。本文综述了人工智能的发展历程、核心技术、应用领域以及未来发展趋势。通过对人工智能的定义、历史背景、主要技术（如机器学习、深度学习、自然语言处理等）的详细介绍，探讨了人工智能在医疗、金融、教育、交通等领域的应用，并分析了人工智能发展过程中面临的挑战与机遇。最后，本文对人工智能的未来发展进行了展望，提出了可能的突破方向。

1. 引言
人工智能是指通过计算机程序模拟人类智能的一门科学。自20世纪50年代诞生以来，人工智能经历了多次起伏，近年来随着计算能力的提升和大数据的普及，人工智能技术取得了显著的进展。人工智能的应用已经渗透到日常生活的方方面面，从智能手机的语音助手到自动驾驶汽车，从医疗诊断到金融分析，人工智能正在改变着人类社会的运行方式。

2.':
🔍 文档 page_content='人工智能的发展历程
2.1 早期发展
人工智能的概念最早可以追溯到20世纪50年代。1956年，达特茅斯会议（Dartmouth Conference）被认为是人工智能研究的正式开端。在随后的几十年里，人工智能研究经历了多次高潮与低谷。早期的研究主要集中在符号逻辑和专家系统上，但由于计算能力的限制和算法的不足，进展缓慢。
2.2 机器学习的兴起
20世纪90年代，随着统计学习方法的引入，机器学习逐渐成为人工智能研究的主流。支持向量机（SVM）、决策树、随机森林等算法在分类和回归任务中取得了良好的效果。这一时期，机器学习开始应用于数据挖掘、模式识别等领域。
2.3 深度学习的突破
2012年，深度学习在图像识别领域取得了突破性进展，标志着人工智能进入了一个新的阶段。深度学习通过多层神经网络模拟人脑的工作方式，能够自动提取特征并进行复杂的模式识别。卷积神经网络（CNN）、循环神经网络（RNN）和长短期记忆网络（LSTM）等深度学习模型在图像处理、自然语言处理、语音识别等领域取得了显著成果。

3. 人工智能的核心技术
3.1 机器学习
机器学习是人工智能的核心技术之一，通过算法使计算机从数据中学习并做出决策。常见的机器学习算法包括监督学习、无监督学习和强化学习。监督学习通过标记数据

### 其他拆分器：HTMLHeaderTextSplitter：Split by HTML header
根据HTML的 标题标签（如`<h1>、<h2>`等） 将文档划分为逻辑分块，同时保留标题的层级结构信息

In [16]:
# 1.导入相关依赖
from langchain_text_splitters import HTMLHeaderTextSplitter
# 2.定义HTML文件
html_string = """
<!DOCTYPE html>
<html>
<body>
<div>
<h1>欢迎来到尚硅谷！</h1>
<p>尚硅谷是专门培训IT技术方向</p>
<div>
<h2>尚硅谷老师简介</h2>
<p>尚硅谷老师拥有多年教学经验，都是从一线互联网下来</p>
<h3>尚硅谷北京校区</h3>
<p>北京校区位于宏福科技园区</p>
</div>
</div>
</body>
</html>
"""
# 4.用于指定要根据哪些HTML标签来分割文本
headers_to_split_on = [
("h1", "标题1"),
("h2", "标题2"),
("h3", "标题3"),
] #
# 5.定义HTMLHeaderTextSplitter分割器
html_splitter = HTMLHeaderTextSplitter(headers_to_split_on=headers_to_split_on)
# 6.分割器分割
html_header_splits = html_splitter.split_text(html_string)
html_header_splits

[Document(metadata={'标题1': '欢迎来到尚硅谷！'}, page_content='欢迎来到尚硅谷！'),
 Document(metadata={'标题1': '欢迎来到尚硅谷！'}, page_content='尚硅谷是专门培训IT技术方向'),
 Document(metadata={'标题1': '欢迎来到尚硅谷！', '标题2': '尚硅谷老师简介'}, page_content='尚硅谷老师简介'),
 Document(metadata={'标题1': '欢迎来到尚硅谷！', '标题2': '尚硅谷老师简介'}, page_content='尚硅谷老师拥有多年教学经验，都是从一线互联网下来'),
 Document(metadata={'标题1': '欢迎来到尚硅谷！', '标题2': '尚硅谷老师简介', '标题3': '尚硅谷北京校区'}, page_content='尚硅谷北京校区'),
 Document(metadata={'标题1': '欢迎来到尚硅谷！', '标题2': '尚硅谷老师简介', '标题3': '尚硅谷北京校区'}, page_content='北京校区位于宏福科技园区')]

### 类型2：CodeTextSplitter：Split code
CodeTextSplitter是一个 专为代码文件 设计的文本分割器（Text Splitter），支持代码的语言包括['cpp',
'go', 'java', 'js', 'php', 'proto', 'python', 'rst', 'ruby', 'rust', 'scala', 'swift', 'markdown', 'latex', 'html','sol']。它能够根据编程语言的语法结构（如函数、类、代码块等）智能地拆分代码，保持代码逻辑的完
整性。

In [17]:
# 1.导入相关依赖
from langchain_text_splitters import (
    Language,
    RecursiveCharacterTextSplitter,
)
from pprint import pprint
# 2.定义要分割的python代码片段
PYTHON_CODE = """
def hello_world():
print("Hello, World!")
def hello_world1():
print("Hello, World1!")
"""
# 3.定义递归字符切分器
python_splitter = RecursiveCharacterTextSplitter.from_language(
language=Language.PYTHON,
chunk_size=50,
chunk_overlap=0
) #
# 4.文档切分
python_docs = python_splitter.create_documents(texts=[PYTHON_CODE])
pprint(python_docs)

[Document(metadata={}, page_content='def hello_world():\nprint("Hello, World!")'),
 Document(metadata={}, page_content='def hello_world1():\nprint("Hello, World1!")')]


### 类型3：MarkdownTextSplitter：md数据类型
因为Markdown格式有特定的语法，一般整体内容由 h1、h2、h3 等多级标题组织，所以MarkdownHeaderTextSplitter的切分策略就是根据 标题来分割文本内容 。

In [17]:
from langchain_text_splitters import MarkdownTextSplitter
markdown_text = """
# 一级标题\n
这是一级标题下的内容\n\n
## 二级标题\n
- 二级下列表项1\n
- 二级下列表项2\n
"""
# 关键步骤：直接修改实例属性
splitter = MarkdownTextSplitter(chunk_size=30, chunk_overlap=0)
splitter._is_separator_regex = True # 强制将分隔符视为正则表达式
# 执行分割
docs = splitter.create_documents(texts = [markdown_text])
# print(len(docs))
print(docs)
for i, doc in enumerate(docs):
    print(f"\n🔍 分块 {i + 1}:")
    print(doc.page_content)

[Document(metadata={}, page_content='# 一级标题\n\n这是一级标题下的内容'), Document(metadata={}, page_content='## 二级标题\n\n- 二级下列表项1\n\n- 二级下列表项2')]

🔍 分块 1:
# 一级标题

这是一级标题下的内容

🔍 分块 2:
## 二级标题

- 二级下列表项1

- 二级下列表项2
